In [16]:
#2021.07.28. WED
#Team_2nn

#00. 패키지 호출
import pandas as pd 
import numpy as np 
import warnings 

#00-1. warning message ignore 
warnings.filterwarnings(action='ignore')

#00-2. 지수표기법을 정수표기법으로 변환하기.
pd.options.display.float_format = '{:.5f}'.format

#00-3. 메인 변수 정의하기.
MAIN     = '가구당인원'
MAIN_ENG = 'PPH'


In [17]:
#01. '@@@@ 네이버스마트스토어 고객프로파일 가구당 인원' 데이터셋 탐색하기. 
#(1) 데이터셋 불러오기. 
customer_data_raw = pd.read_excel('../data/raw/농협양곡 네이버스마트스토어 판매데이터/네이버_상품_고객프로파일_가구당인원_2021-01-01_2021-06-30.xlsx')

#(2) 데이터셋 살펴보기. 
customer_data_raw
print('Eliminated output due to data security')

Eliminated output due to data security


In [18]:
#(3) 데이터셋 결측치 확인하기. 
customer_data_raw.isna().sum()

상품카테고리(대)    0
상품카테고리(중)    0
상품카테고리(소)    0
상품카테고리(세)    0
상품명          0
상품ID         0
가구당인원        0
결제금액         0
결제수          0
결제상품수량       0
환불금액         0
환불건수         0
환불수량         0
dtype: int64

In [19]:
#(4) 컬럼 별 유니크값 확인하기.
#①상품카테고리(대)
customer_data_raw.iloc[:,0].value_counts()

식품    171
Name: 상품카테고리(대), dtype: int64

In [20]:
#②상품카테고리(중)
customer_data_raw.iloc[:,1].value_counts()

농산물    171
Name: 상품카테고리(중), dtype: int64

In [21]:
#③상품카테고리(소)
customer_data_raw.iloc[:,2].value_counts()

쌀         164
잡곡/혼합곡      7
Name: 상품카테고리(소), dtype: int64

In [22]:
#④상품카테고리(세)
customer_data_raw.iloc[:,3].value_counts()

백미     164
혼합곡      7
Name: 상품카테고리(세), dtype: int64

In [23]:
#⑤상품명
customer_data_raw.iloc[:,4].value_counts()
print('Eliminated output due to data security')

Eliminated output due to data security


In [24]:
#⑥메인변수
customer_data_raw.iloc[:,6].value_counts()

2인이상      94
1인        73
(알수없음)     4
Name: 가구당인원, dtype: int64

In [25]:
#(5) 원본 보관하기.
customer_data = customer_data_raw

#(6) 상품명의 '[@@@@]' 스트링 삭제하기.
product_name_list = [] 
for i in range(0,len(customer_data))                               :
    product_name = customer_data['상품명'][i].split()[1:]
    product_name = ' '.join(product_name)
    product_name_list.append(product_name)
customer_data['product_name'] = product_name_list

#(7) cultivar(재배품종) 컬럼 생성하기.
cultivar_list = []
for i in range(len(customer_data))                                 :
    if customer_data['product_name'][i].find('고시히카리') != -1   :
        cultivar = '고시히카리'
    elif customer_data['product_name'][i].find('삼광') != -1       :
        cultivar = '삼광'
    elif customer_data['product_name'][i].find('새청무') != -1     :
        cultivar = '새청무'
    elif customer_data['product_name'][i].find('신동진') != -1     :
        cultivar = '신동진'
    elif customer_data['product_name'][i].find('십리향') != -1     :
        cultivar = '십리향'
    elif customer_data['product_name'][i].find('영호진미') != -1   :
        cultivar = '영호진미'
    elif customer_data['product_name'][i].find('오대') != -1       :
        cultivar = '오대'  
    elif customer_data['product_name'][i].find('일미') != -1       :
        cultivar = '일미'  
    elif customer_data['product_name'][i].find('일품') != -1       :
        cultivar = '일품'  
    elif customer_data['product_name'][i].find('진상') != -1       :
        cultivar = '진상'  
    elif customer_data['product_name'][i].find('참드림') != -1     :
        cultivar = '참드림'  
    elif customer_data['product_name'][i].find('추청') != -1       :
        cultivar = '추청'  
    elif customer_data['product_name'][i].find('하이아미') != -1   :
        cultivar = '하이아미'  
    elif customer_data['product_name'][i].find('혼합') != -1       :
        cultivar = '혼합'  
    else                                                           :
        cultivar = '알수없음'
    cultivar_list.append(cultivar)
customer_data['cultivar'] = cultivar_list

#(8) kakao_yn 컬럼 생성하기
customer_data['kakao_yn'] = customer_data['product_name'].str.find('카카오')
customer_data['kakao_yn'][customer_data['kakao_yn'] == -1] = 'no'
customer_data['kakao_yn'][customer_data['kakao_yn'] == 0] = 'yes'

#(9) Actual Transactions Quantity(실질거래수량) 컬럼 생성하기.
customer_data['ATQ'] = customer_data['결제상품수량'] - customer_data['환불수량']

#(10) Actual transaction amount(실질거래금액)
customer_data['ATA'] = customer_data['결제금액'] - customer_data['환불금액']

#(11) unit_weight(kg)(제품의 단위 무게) 컬럼 생성하기. 
rice_weight_list = []
for  i in range(0,len(customer_data))                                                                              :
    if customer_data['상품명'][i].find('+') == -1 and customer_data['상품명'][i].find('X') == -1                   : 
        if customer_data['상품명'][i].split()[-1].endswith('kg')                                                   :
            rice_weight = int(customer_data['상품명'].str.split()[i][-1].replace('kg',''))
            rice_weight_list.append(rice_weight)
        elif customer_data['상품명'][i].split()[-2].endswith('kg')                                                 :
            rice_weight = int(customer_data['상품명'].str.split()[i][-2].replace('kg',''))
            rice_weight_list.append(rice_weight)

    else                                                                                                           :
        for k in range(0,len(customer_data['상품명'][i].split()))                                                  :
            if customer_data['상품명'][i].split()[k].endswith('kg') and customer_data['상품명'][i].find('+') != -1 :
                rice_weight = int(customer_data['상품명'][i].split()[k].split('+')[0].replace('kg','')) *2
                rice_weight_list.append(rice_weight)
                break
            if customer_data['상품명'][i].split()[k].endswith('kg') and customer_data['상품명'][i].find('+') == -1 :
                rice_weight = int(customer_data['상품명'][i].split()[k].replace('kg','')) *2
                rice_weight_list.append(rice_weight)
                break
customer_data['unit_weight(kg)'] = rice_weight_list

#(12) total sales unit(총 판매 단위) 컬럼 생성하기.
customer_data['TSU'] = customer_data['ATQ'] * customer_data['unit_weight(kg)']

#(13) price per unit weight(단위 무게당 가격) 컬럼 생성하기.
customer_data['PPU'] = customer_data['ATA'] / (customer_data['ATQ'] * customer_data['unit_weight(kg)'])

#(14) 메인 컬럼의 (알수없음) 로우 삭제하기.
customer_data = customer_data[customer_data[MAIN] != '(알수없음)'] 

#(15) 실질거래금액이 0인 로우 삭제하기.
customer_data = customer_data[customer_data['ATA'] != 0]

#(16) 불필요 컬럼 삭제하기.
del customer_data['상품카테고리(대)']
del customer_data['상품카테고리(중)']
del customer_data['상품카테고리(소)']
del customer_data['상품ID']
del customer_data['결제수']
del customer_data['환불건수']
del customer_data['상품명']

#(17) 데이터셋 컬럼 위치 조정하기. 
customer_data = customer_data[[
    'product_name','상품카테고리(세)','cultivar','kakao_yn','결제상품수량','결제금액','환불수량','환불금액','ATQ','ATA','unit_weight(kg)','TSU','PPU', MAIN
]]

#(18) PPU 변수 0의 자리에서 반올림후 정수처리하기. 
customer_data['PPU'] = round(customer_data['PPU'],0).astype(int)

#(19) 데이터셋 컬럼 이름 변경하기.
customer_data = customer_data.rename(columns={
    '상품카테고리(세)' : 'product_category',
    '결제상품수량'     : 'QOPP',                #Quantity of payment products
    '결제금액'         : 'AOP',                 #Amount of payment
    '환불수량'         : 'RQ',                  #Refund quantity
    '환불금액'         : 'RA',                  #Refund amount
    MAIN               : MAIN_ENG               #People per household
})

#(21) 데이터셋 확인하기.
customer_data
print('Eliminated output due to data security')

Eliminated output due to data security


In [26]:
#02. 품종의 알수없음 로우 처리하기. 
#(1) 객체 지정하기. 
problem_data = customer_data[customer_data['cultivar']=='알수없음']

#(2) 알수없음 컬럼의 유니크값 파악하기. 
problem_data['product_name'].value_counts().to_frame().reset_index().head(15)
print('Eliminated output due to data security')

Eliminated output due to data security


In [27]:
#(3) 문제 데이터셋 저장하기. 
problem_data['product_name'].value_counts().to_frame().reset_index().to_excel('../data/problem_data.xlsx', index=False)

#(4) 문제 데이터셋 인터넷 처리 후 저장하기. 
problem_data_modi = pd.read_excel('../data/problem_data_modi.xlsx')

#(5) 문제 데이터셋과 기존 데이터셋 결합하기. 
customer_data = pd.merge(customer_data, problem_data_modi, left_on='product_name', right_on='index', how='left')

#(6) 기존 데이터셋의 cultivar 컬럼에서 알수없음 값 대체하기.
for i in range(len(customer_data)) :
    if customer_data['cultivar_x'][i] == '알수없음' :
        customer_data['cultivar_x'][i] = customer_data['cultivar_y'][i]

#(7) 불필요 컬럼 삭제하기.
del customer_data['index']
del customer_data['cultivar_y']

#(8) 데이터셋 컬럼 이름 변경하기.
customer_data = customer_data.rename(columns={'cultivar_x':'cultivar'})

#(9) 데이터셋 저장하기. 
customer_data.to_excel(f'../data/preprocessing/customer_data_{MAIN_ENG}.xlsx', index=False)

#(10) 데이터셋 확인하기.
customer_data
print('Eliminated output due to data security')

Eliminated output due to data security


In [28]:
#(11) 처리 재확인하기.
customer_data['cultivar'].value_counts()

혼합       29
추청       29
삼광       25
신동진      24
고시히카리    10
영호진미      9
일미        9
일품        8
참드림       5
하이아미      4
진상        4
십리향       4
오대        3
새청무       2
해담        1
Name: cultivar, dtype: int64